# Notebook 05 — Evaluation & Ablation Study

This notebook answers a critical question: **how much does each detection layer contribute
to the overall system?**

We perform:
1. **Baseline evaluation** — full model with all 7 features
2. **Ablation study** — remove one layer at a time and measure the impact
3. **False positive analysis** — examine which benign packages get flagged
4. **Aggregation weight sensitivity** — how the 5-layer weighted scoring behaves
5. **Limitations discussion** — honest assessment of the system's boundaries

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix,
)
from xgboost import XGBClassifier

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

cwd = Path.cwd()
candidates = [cwd, cwd.parent, cwd / "supply-chain-detector"]
PROJECT_ROOT = next((p for p in candidates if (p / "detector").exists()), cwd)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from detector.aggregator import aggregate_risk, AggregationWeights
from detector.classifier import DEFAULT_FEATURE_NAMES

DATA_DIR = PROJECT_ROOT / "data" / "processed"
CACHE_DIR = DATA_DIR / "notebook_cache"
SPLITS_DIR = DATA_DIR / "splits"
FEATURE_NAMES = list(DEFAULT_FEATURE_NAMES)
print(f"Features: {FEATURE_NAMES}")

In [ ]:
# Load pre-computed feature matrices from cache (instant)
if (CACHE_DIR / "features_train.npz").exists():
    _tr = np.load(CACHE_DIR / "features_train.npz", allow_pickle=True)
    _va = np.load(CACHE_DIR / "features_val.npz", allow_pickle=True)
    _te = np.load(CACHE_DIR / "features_test.npz", allow_pickle=True)
    X_train, y_train = _tr["X"], _tr["y"]
    X_val, y_val = _va["X"], _va["y"]
    X_test, y_test = _te["X"], _te["y"]
    test_names = list(_te["names"])
    print("Loaded features from pre-computed cache.")
else:
    # Fallback: compute on the fly
    from detector.layer1_metadata.metadata_analyzer import analyze_metadata_risk
    from detector.layer3_static.static_analyzer import analyze_static_risk

    def _dep_count(rec):
        n = 0
        if isinstance(rec.get("dependencies"), dict): n += len(rec["dependencies"])
        if isinstance(rec.get("requires_dist"), list): n += len(rec["requires_dist"])
        return n

    def extract_features(rec):
        name = str(rec.get("package_name","")).strip().lower()
        reg = str(rec.get("registry","")).strip().lower()
        src = str(rec.get("source_code",""))
        dc = _dep_count(rec)
        meta = float(analyze_metadata_risk(name, reg, rec).get("final_score", 0.0))
        stat = float(analyze_static_risk(src).get("final_score", 0.0)) if src.strip() else 0.0
        emb = (25.0 if len(src) < 200 else 10.0) if src.strip() else 0.0
        graph = min(float(dc) * 5.0, 100.0)
        return [meta, emb, stat, graph, float(len(name)), float(dc), float(0 if rec.get("author") else 1)]

    def build_matrix(records):
        X, y, names = [], [], []
        for r in records:
            label = str(r.get("label","")).strip().lower()
            if label not in {"benign","malicious"}: continue
            X.append(extract_features(r))
            y.append(1 if label == "malicious" else 0)
            names.append(r.get("package_name",""))
        return np.array(X, dtype=np.float32), np.array(y, dtype=np.int32), names

    print("Cache not found — extracting features...")
    train_recs = json.loads((SPLITS_DIR/"train.json").read_text(encoding="utf-8"))
    val_recs = json.loads((SPLITS_DIR/"val.json").read_text(encoding="utf-8"))
    test_recs = json.loads((SPLITS_DIR/"test.json").read_text(encoding="utf-8"))
    X_train, y_train, _ = build_matrix(train_recs)
    X_val, y_val, _ = build_matrix(val_recs)
    X_test, y_test, test_names = build_matrix(test_recs)

print(f"Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}")

## 1. Baseline — Full Model

Train and evaluate with all 7 features to establish the performance ceiling.

In [ ]:
def train_and_evaluate(X_tr, y_tr, X_te, y_te, label=""):
    """Train XGBoost and return test metrics."""
    n_neg = int((y_tr == 0).sum())
    n_pos = int((y_tr == 1).sum())
    spw = n_neg / n_pos if n_pos > 0 else 1.0

    clf = XGBClassifier(
        n_estimators=180, max_depth=4, learning_rate=0.08,
        subsample=0.9, colsample_bytree=0.9,
        objective="binary:logistic", eval_metric="logloss",
        random_state=42, scale_pos_weight=spw,
    )
    clf.fit(X_tr, y_tr, verbose=False)

    proba = clf.predict_proba(X_te)[:, 1]
    y_pred = (proba >= 0.5).astype(int)
    return {
        "config": label,
        "precision": precision_score(y_te, y_pred, zero_division=0),
        "recall":    recall_score(y_te, y_pred, zero_division=0),
        "f1":        f1_score(y_te, y_pred, zero_division=0),
        "roc_auc":   roc_auc_score(y_te, proba) if len(np.unique(y_te)) > 1 else 0.0,
    }, clf, proba


baseline, baseline_clf, baseline_proba = train_and_evaluate(
    X_train, y_train, X_test, y_test, label="All features"
)

print("BASELINE (all 7 features):")
for k, v in baseline.items():
    if k != "config":
        print(f"  {k:12s}: {v:.4f}")

## 2. Ablation Study — Remove One Layer at a Time

We systematically zero-out features from each detection layer and retrain the model.
The performance drop reveals each layer's contribution.

| Ablation | Features zeroed |
|----------|----------------|
| −Metadata | `metadata_score` |
| −Embeddings | `embedding_score` |
| −Static | `static_score` |
| −Graph | `graph_score`, `dep_count` |
| −Identity | `name_length`, `author_missing` |

In [ ]:
# Define which feature indices to zero for each ablation group
ablation_groups = {
    "\u2212 Metadata":   [0],       # metadata_score
    "\u2212 Embeddings": [1],       # embedding_score
    "\u2212 Static":     [2],       # static_score
    "\u2212 Graph":      [3, 5],    # graph_score, dep_count
    "\u2212 Identity":   [4, 6],    # name_length, author_missing
}

ablation_results = [baseline]

for group_name, zero_indices in ablation_groups.items():
    # Zero out the specified features
    X_tr_abl = X_train.copy()
    X_te_abl = X_test.copy()
    for idx in zero_indices:
        X_tr_abl[:, idx] = 0.0
        X_te_abl[:, idx] = 0.0

    result, _, _ = train_and_evaluate(X_tr_abl, y_train, X_te_abl, y_test, label=group_name)
    ablation_results.append(result)
    zeroed_names = [FEATURE_NAMES[i] for i in zero_indices]
    print(f"{group_name:20s}  F1={result['f1']:.4f}  AUC={result['roc_auc']:.4f}  (zeroed: {zeroed_names})")

df_ablation = pd.DataFrame(ablation_results)
df_ablation

In [ ]:
# Ablation impact visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, metric, color in [
    (axes[0], "f1", "#e74c3c"),
    (axes[1], "roc_auc", "#3498db"),
]:
    values = df_ablation[metric].values
    labels = df_ablation["config"].values
    colors = ["#2ecc71" if i == 0 else color for i in range(len(values))]

    bars = ax.barh(labels, values, color=colors, edgecolor="white")
    ax.axvline(x=values[0], color="#2ecc71", linestyle="--", alpha=0.5, label="Baseline")
    ax.set_xlabel(metric.upper())
    ax.set_title(f"Ablation Study — {metric.upper()}", fontweight="bold")

    for bar, val in zip(bars, values):
        ax.text(val + 0.005, bar.get_y() + bar.get_height() / 2,
                f"{val:.3f}", va="center", fontsize=9)

plt.tight_layout()
plt.show()

# Delta from baseline
print("\nDelta from Baseline:")
print(f"{'Config':20s} {'F1 delta':>10s} {'AUC delta':>10s}")
print("-" * 42)
for _, row in df_ablation.iterrows():
    f1_d  = row["f1"] - baseline["f1"]
    auc_d = row["roc_auc"] - baseline["roc_auc"]
    print(f"{row['config']:20s} {f1_d:>+10.4f} {auc_d:>+10.4f}")

## 3. False Positive Analysis

Which benign packages does the full model incorrectly flag as malicious?
Understanding false positives helps refine detection thresholds.

In [ ]:
# Find false positives and false negatives
pred_labels = (baseline_proba >= 0.5).astype(int)

fp_indices = np.where((pred_labels == 1) & (y_test == 0))[0]
fn_indices = np.where((pred_labels == 0) & (y_test == 1))[0]

print(f"False Positives (benign flagged as malicious): {len(fp_indices)}")
print(f"False Negatives (malicious missed):            {len(fn_indices)}")

if len(fp_indices) > 0:
    print("\n--- False Positives ---")
    fp_data = []
    for idx in fp_indices[:10]:  # Show up to 10
        fp_data.append({
            "package": test_names[idx],
            "predicted_prob": f"{baseline_proba[idx]:.3f}",
            **{FEATURE_NAMES[j]: X_test[idx, j] for j in range(len(FEATURE_NAMES))},
        })
    display(pd.DataFrame(fp_data))

if len(fn_indices) > 0:
    print("\n--- False Negatives ---")
    fn_data = []
    for idx in fn_indices[:10]:
        fn_data.append({
            "package": test_names[idx],
            "predicted_prob": f"{baseline_proba[idx]:.3f}",
            **{FEATURE_NAMES[j]: X_test[idx, j] for j in range(len(FEATURE_NAMES))},
        })
    display(pd.DataFrame(fn_data))

## 4. Aggregation Weight Sensitivity

In production, the final risk score is computed by `detector/aggregator.py` using a
**weighted combination** of all five layer scores, independent of the classifier.

Default weights:
- Metadata: 0.22 | Embedding: 0.15 | Static: 0.25 | LLM: 0.18 | Graph: 0.15 | Classifier: 0.05

Below we visualize how varying the metadata weight affects the aggregated scores.

In [ ]:
# Simulate aggregation with different weight profiles
scenarios = [
    {"name": "Default",         "metadata": 0.22, "embedding": 0.15, "static": 0.25, "llm": 0.18, "graph": 0.15, "classifier": 0.05},
    {"name": "Metadata-heavy",  "metadata": 0.40, "embedding": 0.10, "static": 0.20, "llm": 0.15, "graph": 0.10, "classifier": 0.05},
    {"name": "Static-heavy",    "metadata": 0.15, "embedding": 0.10, "static": 0.40, "llm": 0.20, "graph": 0.10, "classifier": 0.05},
    {"name": "Balanced",        "metadata": 0.18, "embedding": 0.18, "static": 0.18, "llm": 0.18, "graph": 0.18, "classifier": 0.10},
]

# Test with a known-safe and known-suspicious package profile
test_profiles = [
    {"label": "Safe (numpy)",     "meta": 0,  "emb": 10, "static": 5,  "llm": 0, "graph": 20, "clf": 5},
    {"label": "Suspicious (typosquat)", "meta": 85, "emb": 60, "static": 45, "llm": 70, "graph": 10, "clf": 75},
    {"label": "Ambiguous (new pkg)",    "meta": 40, "emb": 35, "static": 20, "llm": 15, "graph": 5,  "clf": 45},
]

rows = []
for scenario in scenarios:
    weights = AggregationWeights(
        metadata=scenario["metadata"], embedding=scenario["embedding"],
        static=scenario["static"], llm=scenario["llm"],
        graph=scenario["graph"], classifier=scenario["classifier"],
    )
    for profile in test_profiles:
        result = aggregate_risk(
            metadata_score=profile["meta"], embedding_score=profile["emb"],
            static_score=profile["static"], llm_score=profile["llm"],
            graph_score=profile["graph"], classifier_score=profile["clf"],
            weights=weights, llm_was_triggered=True,
        )
        rows.append({
            "weight_profile": scenario["name"],
            "package_profile": profile["label"],
            "final_score": result["final_score"],
            "decision": result["decision"],
            "consensus": result["consensus_signals"],
        })

df_agg = pd.DataFrame(rows)
pivot = df_agg.pivot(index="package_profile", columns="weight_profile", values="final_score")

fig, ax = plt.subplots(figsize=(10, 4))
pivot.plot(kind="bar", ax=ax, edgecolor="white")
ax.set_ylabel("Aggregated Risk Score")
ax.set_title("Aggregation Sensitivity to Weight Profiles", fontweight="bold")
ax.axhline(y=50, color="orange", linestyle="--", alpha=0.5, label="Review threshold")
ax.axhline(y=80, color="red", linestyle="--", alpha=0.5, label="Block threshold")
ax.legend(fontsize=8)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print("\nDecision matrix:")
print(df_agg.pivot(index="package_profile", columns="weight_profile", values="decision"))

## 5. Limitations & Discussion

### What works well
- **Metadata-based detection** (typosquatting, author analysis) is highly effective for
  the most common supply-chain attack vector: name impersonation.
- **Multi-layer architecture** ensures no single point of failure — even if one layer is
  bypassed, others can catch the threat.
- **Consensus mechanism** boosts confidence when multiple layers agree.

### Known limitations

| Limitation | Impact | Mitigation |
|-----------|--------|------------|
| Small dataset (~430 records) | Risk of overfitting, limited generalization | `scale_pos_weight`, conservative hyperparameters |
| Source code mostly absent offline | `static_score`/`embedding_score` underutilized | Live pipeline fetches tarballs on demand |
| Typosquat-heavy malicious set | Detection biased toward name-squatting attacks | Expand with dependency confusion, account takeover samples |
| LLM layer not in offline eval | Can't measure LLM contribution in ablation | Tested separately via live API (see live scan results) |
| No concept drift monitoring | Model may degrade as attack patterns evolve | Schedule periodic retraining with fresh data |

### Future work
1. **Expand the dataset** — incorporate more attack types (dependency confusion, protestware,
   account takeover)
2. **Online learning** — update the model incrementally as new scans are performed
3. **Threshold optimization** — tune the 50/80 decision boundaries via precision-recall curves
4. **Cross-registry transfer** — test if a model trained on PyPI generalizes to npm and vice versa

In [ ]:
# Final summary table
print("=" * 60)
print(" EVALUATION SUMMARY")
print("=" * 60)
print(f"\n{'Metric':15s} {'Baseline':>10s}")
print("-" * 27)
for metric in ["precision", "recall", "f1", "roc_auc"]:
    print(f"{metric:15s} {baseline[metric]:>10.4f}")

print(f"\n\nMost impactful layer (largest F1 drop when removed):")
drops = []
for _, row in df_ablation.iterrows():
    if row["config"] != "All features":
        drops.append((row["config"], baseline["f1"] - row["f1"]))
drops.sort(key=lambda x: x[1], reverse=True)
for name, drop in drops:
    direction = "\u2193" if drop > 0 else "\u2191"
    print(f"  {name:20s}  F1 {direction} {abs(drop):.4f}")

print(f"\nFalse Positives: {len(fp_indices)} / {int((y_test == 0).sum())} benign")
print(f"False Negatives: {len(fn_indices)} / {int((y_test == 1).sum())} malicious")
print("\nDone.")